In [19]:
import pandas as pd
import matplotlib as plt
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import importlib
import preprocessing as p
importlib.reload(p)

from sklearn.model_selection import GridSearchCV
from preprocessing import ordinal_scales
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, PowerTransformer

In [12]:
df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')

X=df_train.drop(columns=['SalePrice'])
y=df_train['SalePrice']
X_train,X_valid,y_train,y_valid=train_test_split(X,y,test_size=0.2,random_state=42)

test_ids = df_test['Id'].copy()

In [13]:
modeles = {
    'LinearRegression': LinearRegression(),
    'Ridge':            Ridge(alpha=10.0),
    'Lasso':            Lasso(alpha=0.001),
    'ElasticNet':       ElasticNet(alpha=0.001, l1_ratio=0.5),
    'RandomForest':     RandomForestRegressor(n_estimators=300, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'KNN':              KNeighborsRegressor(n_neighbors=10),
}

In [ ]:
def features(df):
    df=df.copy()
    df['TotalSF']=df['TotalBsmtSF']+df['GrLivArea']
    df['RatioArea']=df['TotalBsmtSF']/df['LotArea']
    df['RoomSize']=df['GrLivArea']/df['TotRmsAbvGrd']
    df['BedroomSize']=df['BedroomAbvGr']/df['GrLivArea']
    df['HouseAge']=df['YrSold']-df['YearBuilt']
    df['HouseRemod']=df['YrSold']-df['YearRemodAdd']
    return df

from sklearn.preprocessing import FunctionTransformer

feature_eng = FunctionTransformer(features)

In [21]:
X=features(X)

In [22]:
ordinal_cols = list(ordinal_scales.keys())

num_cols = X.select_dtypes(include='number').columns.tolist()

# toutes les colonnes texte/bool, MOINS les ordinales
cat_cols = X.select_dtypes(include=['bool', 'object']).columns.tolist()
nominal_cols = [c for c in cat_cols if c not in ordinal_cols]

num_cols = [c for c in num_cols if c != 'Id']

In [23]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('power', PowerTransformer(method='yeo-johnson')),
])

ordinal_pipe= Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='None')),
('OrdinalEncoder',OrdinalEncoder(categories=[ordinal_scales[c] for c in ordinal_cols]))])

nominal_pipe = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='None')),('OneHotEncoder',OneHotEncoder(handle_unknown='ignore', sparse_output=False))])


In [24]:
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('ord', ordinal_pipe, ordinal_cols),
    ('nom', nominal_pipe, nominal_cols),
])

In [25]:
lasso_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('model', Lasso(max_iter=10000)),   # max_iter élevé pour éviter le warning de convergence
])

param_grid_lasso = {'model__alpha': [0.0001, 0.001, 0.005, 0.01, 0.05]}


grid = GridSearchCV(lasso_pipe, param_grid_lasso, cv=5,
                    scoring='neg_mean_squared_error')
grid.fit(X, np.log1p(y))

print("meilleur alpha :", grid.best_params_)
print("RMSE           :", np.sqrt(-grid.best_score_))

meilleur alpha : {'model__alpha': 0.001}
RMSE           : 0.1251821713113441


In [27]:
df_test=features(df_test)

In [28]:
predictions_log = grid.predict(df_test)
predictions = np.expm1(predictions_log)   # retour en dollars
soumission = pd.DataFrame({ 'Id': test_ids,
'SalePrice': predictions
})
soumission.to_csv('soumission_withFE_Lasso.csv', index=False)